# Registrierungsschritte visualisieren (Opening Alignment)

Dieses Notebook zeigt die vier Schritte aus `create_opening_meshes` explizit an einem Opening:
1. Punkte der Opening-Kontur in die lokale Ebene projizieren
2. Punkte clockwise sortieren
3. 2D-Polygon triangulieren
4. Triangulierte Vertices/Faces auf die Original-Vertices zurueckmappen

Referenz-Code: `ghd/fitting/registration.py` und `utils/utils_registration.py`.

In [ ]:
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

import shapely.geometry
import trimesh

from ghd.fitting.registration import RegistrationwOpeningAlignment
from utils import utils_registration as u_register


def set_equal_3d_axes(ax, points):
    points = np.asarray(points)
    p_min = points.min(axis=0)
    p_max = points.max(axis=0)
    center = 0.5 * (p_min + p_max)
    radius = 0.5 * np.max(p_max - p_min)
    radius = max(radius, 1e-6)
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_zlim(center[2] - radius, center[2] + radius)


def add_triangle_collection(ax, vertices, faces, color='tab:orange', alpha=0.8, linewidth=0.4):
    tri_verts = vertices[np.asarray(faces, dtype=np.int64)]
    tri = Poly3DCollection(
        tri_verts,
        facecolors=color,
        edgecolors='k',
        linewidths=linewidth,
        alpha=alpha,
    )
    ax.add_collection3d(tri)


def discover_cases(root_dir, mesh_filename='part_aligned.obj'):
    root_dir = Path(root_dir)
    cases = []
    for p in sorted(root_dir.iterdir()):
        if p.is_dir() and (p / mesh_filename).exists():
            cases.append(p.name)
    return cases


def triangulate_polygon_with_fallback(polygon, triangle_args='p'):
    # Try available triangulation backends in a deterministic order.
    attempts = [
        ('earcut', {'engine': 'earcut'}),
        ('triangle', {'engine': 'triangle'}),
        ('default', {}),
    ]
    errors = []
    for name, kwargs in attempts:
        try:
            result = trimesh.creation.triangulate_polygon(
                polygon,
                triangle_args=triangle_args,
                **kwargs,
            )
            return result, name
        except Exception as exc:
            errors.append((name, str(exc)))

    lines = [
        'Keine Triangulations-Engine verfuegbar.',
        'Installiere eine Engine, z. B.:',
        '  pip install mapbox-earcut',
        'oder',
        '  pip install triangle',
        'Fehler pro Versuch:',
    ]
    for name, err in errors:
        lines.append(f'  - {name}: {err}')
    raise RuntimeError('\n'.join(lines))


In [ ]:
# Konfiguration
ROOT_DIR = Path('./checkpoints/alignment')
MESH_FILENAME = 'part_aligned.obj'
TARGET_CASE = ''  # leer = automatisch erster valider Case
NUM_OP = 3
OPENING_IDX = 0

cases = discover_cases(ROOT_DIR, MESH_FILENAME)
if not cases:
    raise RuntimeError(f'Keine Cases mit {MESH_FILENAME} unter {ROOT_DIR} gefunden.')

if not TARGET_CASE:
    TARGET_CASE = cases[0]

if TARGET_CASE not in cases:
    raise ValueError(f'TARGET_CASE={TARGET_CASE} nicht gefunden. Beispiele: {cases[:10]}')

print('Anzahl Cases:', len(cases))
print('Gewaehlter Case:', TARGET_CASE)


In [ ]:
# Registrierung laden und Openings automatisch finden
args = SimpleNamespace(
    device='cpu',
    opening_registration_mode='auto',
    opening_auto_min_vertices=8,
    opening_auto_area_ratio=0.01,
    centreline_filename='Centerline model.vtk',
)

reg = RegistrationwOpeningAlignment(
    args=args,
    root=str(ROOT_DIR),
    target=TARGET_CASE,
    num_op=NUM_OP,
    suffix='.obj',
)
reg.register_openings_auto()

print('Gefundene Openings:', len(reg.op_v_indices))
for i, v_idx in enumerate(reg.op_v_indices):
    print(f'  Opening {i}: {len(v_idx)} Vertices')

if OPENING_IDX < 0 or OPENING_IDX >= len(reg.op_v_indices):
    raise IndexError(f'OPENING_IDX={OPENING_IDX} ausserhalb [0, {len(reg.op_v_indices)-1}]')

op_indices = np.asarray(reg.op_v_indices[OPENING_IDX], dtype=np.int64)
op_coords = np.asarray(reg.op_v_coords[OPENING_IDX], dtype=np.float64)
op_normal = np.asarray(reg.op_n_mean[OPENING_IDX], dtype=np.float64)

print('Ausgewaehltes Opening:', OPENING_IDX)
print('Konturpunkte:', op_coords.shape[0])


## Schritt 1: Projection in lokale Ebene

In [ ]:
# Exakt wie in create_opening_meshes: projizieren, Path3D bauen, nach 2D wechseln
points_projected = u_register.pcd_to_approx_plane(op_coords, op_normal)
point_sequence = np.concatenate((np.arange(points_projected.shape[0]), np.array([0])))
points_projected_closed = np.concatenate((points_projected, points_projected[0:1]), axis=0)

path_topo = trimesh.path.entities.Entity(point_sequence)
path3D = trimesh.path.path.Path3D([path_topo], vertices=points_projected_closed)
path2D, to_3D = path3D.to_planar()
unsorted_2d = np.asarray(path2D.vertices)

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(op_coords[:, 0], op_coords[:, 1], op_coords[:, 2], s=22, color='tab:blue', label='Original opening points')
ax1.scatter(points_projected[:, 0], points_projected[:, 1], points_projected[:, 2], s=12, color='tab:red', label='Projected points')
for i, p in enumerate(points_projected):
    if i < 25:
        ax1.text(p[0], p[1], p[2], str(i), fontsize=7)
ax1.set_title('3D: Original + Projektion')
ax1.legend(loc='upper right')
set_equal_3d_axes(ax1, points_projected)

ax2 = fig.add_subplot(122)
ax2.scatter(unsorted_2d[:, 0], unsorted_2d[:, 1], s=20, color='tab:blue')
for i, p in enumerate(unsorted_2d):
    if i < 40:
        ax2.text(p[0], p[1], str(i), fontsize=7)
ax2.set_title('2D Punkte (unsortiert)')
ax2.set_aspect('equal', 'box')
plt.tight_layout()
plt.show()


## Schritt 2: Clockwise sortieren und Polygon erzeugen

In [ ]:
sorted_2d = u_register.clock_sort_2D_points(unsorted_2d.copy())
polygon = shapely.geometry.Polygon(sorted_2d)

if not polygon.is_valid:
    print('Warnung: Polygon ist nicht valide (self-intersection moeglich).')
print('Polygon area (2D):', float(polygon.area))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(unsorted_2d[:, 0], unsorted_2d[:, 1], s=18, color='tab:gray')
ax1.plot(np.r_[unsorted_2d[:, 0], unsorted_2d[0, 0]], np.r_[unsorted_2d[:, 1], unsorted_2d[0, 1]], color='tab:gray')
ax1.set_title('Unsortierte Reihenfolge')
ax1.set_aspect('equal', 'box')

ax2.scatter(sorted_2d[:, 0], sorted_2d[:, 1], s=18, color='tab:green')
ax2.plot(np.r_[sorted_2d[:, 0], sorted_2d[0, 0]], np.r_[sorted_2d[:, 1], sorted_2d[0, 1]], color='tab:green')
ax2.set_title('Clockwise sortiert')
ax2.set_aspect('equal', 'box')

plt.tight_layout()
plt.show()


## Schritt 3: Triangulation des 2D-Polygons und Rueckprojektion nach 3D

In [ ]:
(tri_result, triangulation_engine) = triangulate_polygon_with_fallback(
    polygon,
    triangle_args='p',
)
vertices_2d, faces = tri_result
vertices_2d = np.asarray(vertices_2d, dtype=np.float64)
faces = np.asarray(faces, dtype=np.int64)

# Gleiche Nachbearbeitung wie im Projektcode
if faces.min() == 1:
    faces = faces - 1
vertices_2d = vertices_2d[:-1, :]
points_projected_trim = points_projected_closed[:-1, :]

vertices_3d = u_register.trimesh_points_2d_to_3d(vertices_2d, to_3D)

print('Triangulation:')
print('  engine:', triangulation_engine)
print('  vertices_2d:', vertices_2d.shape)
print('  faces:', faces.shape)

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(121)
ax1.triplot(vertices_2d[:, 0], vertices_2d[:, 1], faces, color='k', linewidth=0.7)
ax1.scatter(vertices_2d[:, 0], vertices_2d[:, 1], s=10, color='tab:orange')
ax1.set_title('2D Triangulation')
ax1.set_aspect('equal', 'box')

ax2 = fig.add_subplot(122, projection='3d')
add_triangle_collection(ax2, vertices_3d, faces, color='tab:orange', alpha=0.85)
ax2.scatter(vertices_3d[:, 0], vertices_3d[:, 1], vertices_3d[:, 2], s=10, color='k')
ax2.set_title('Triangulierte Flaeche in 3D')
set_equal_3d_axes(ax2, vertices_3d)

plt.tight_layout()
plt.show()


## Schritt 4: Rueckrechnen auf Original-Vertices

In [ ]:
op_rec_v_indices_map, op_rec_f_map = u_register.get_mapped_sequence_and_faces(
    op_indices,
    points_projected_trim,
    vertices_3d,
    faces,
)

mesh_vertices = np.asarray(reg.mesh_target.vertices)
mesh_faces = np.asarray(reg.mesh_target.triangles)
mapped_vertices_3d = mesh_vertices[op_rec_v_indices_map]

dist = np.linalg.norm(mapped_vertices_3d - vertices_3d, axis=1)
print('Mapping-Qualitaet (Distanz trianguliert -> gemappt):')
print('  mean =', float(dist.mean()))
print('  max  =', float(dist.max()))
print('  mapped vertex ids (erste 20):', op_rec_v_indices_map[:20])

fig = plt.figure(figsize=(14, 6))
ax1 = fig.add_subplot(121, projection='3d')
add_triangle_collection(ax1, vertices_3d, faces, color='tab:orange', alpha=0.8)
ax1.scatter(vertices_3d[:, 0], vertices_3d[:, 1], vertices_3d[:, 2], s=10, color='k')
ax1.set_title('Triangulierte Patch-Vertices (lokal)')
set_equal_3d_axes(ax1, vertices_3d)

ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_trisurf(
    mesh_vertices[:, 0],
    mesh_vertices[:, 1],
    mesh_vertices[:, 2],
    triangles=mesh_faces,
    color='lightgray',
    alpha=0.18,
    edgecolor='none',
)
add_triangle_collection(ax2, mesh_vertices, op_rec_f_map, color='tab:red', alpha=0.95)
ax2.scatter(
    mapped_vertices_3d[:, 0],
    mapped_vertices_3d[:, 1],
    mapped_vertices_3d[:, 2],
    s=12,
    color='tab:blue',
)
ax2.set_title('Rueckgemappte Faces auf Original-Mesh')
set_equal_3d_axes(ax2, mesh_vertices)

plt.tight_layout()
plt.show()


## Optionaler Check gegen Klassenmethode

In [ ]:
# Verifikation: identisch zur internen create_opening_meshes-Implementierung
reg.create_opening_meshes(viz=False)

auto_idx_map = np.asarray(reg.op_rec_v_indices_map[OPENING_IDX])
auto_f_map = np.asarray(reg.op_rec_f_map[OPENING_IDX])

print('Indices identisch:', np.array_equal(auto_idx_map, op_rec_v_indices_map))
print('Faces identisch:  ', np.array_equal(auto_f_map, op_rec_f_map))

if not np.array_equal(auto_idx_map, op_rec_v_indices_map):
    print('Hinweis: Bei unterschiedlichen Reihenfolgen ist meist der Sortier-/Pfadschritt anders.')
